# RISS Inference Demo

This notebook provides a lightweight example for running inference with pretrained RISS weights. It is intended to demonstrate the workflow:

1. check the environment;
2. locate pretrained weights;
3. prepare an example low-energy CEM or mammography input;
4. run inference with the repository `test.py` entry point;
5. visualize and save the synthetic recombined image.

This notebook is **not** intended to reproduce all manuscript-level quantitative results. Full evaluation requires the complete test cohorts, standardized preprocessing, and the evaluation scripts described in the README.

## 1. Environment Check

Run this cell first to confirm that PyTorch and CUDA are available. CPU inference may work for debugging but is usually slow.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import torch
from PIL import Image
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

## 2. Set Repository Paths

The notebook assumes the following repository layout:

```text
RISS/
  test.py
  Checkpoints/le_re/latest_net_G.pth
  notebooks/RISS_inference_demo.ipynb
  notebooks/assets/example/sample_original.png
```

If your checkpoint is stored elsewhere, update `checkpoint_dir` or `experiment_name` below.

In [ ]:
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

test_py = repo_root / "test.py"
checkpoint_dir = repo_root / "Checkpoints"
experiment_name = "le_re"
weights_path = checkpoint_dir / experiment_name / "latest_net_G.pth"

# Raw original mammography image (single-channel low-energy input, not yet aligned).
# Bundled with the notebook so the demo is self-contained.
original_image = repo_root / "notebooks" / "assets" / "example" / "sample_original.png"
demo_dataset = repo_root / "Datasets" / "notebook_demo"
results_dir = repo_root / "results" / "notebook_demo"

print("Repository root:", repo_root)
print("test.py exists:", test_py.exists(), test_py)
print("Weights path:", weights_path)
print("Weights exist:", weights_path.exists())
print("Original image:", original_image)
print("Original image exists:", original_image.exists())

## 3. Preprocess the Raw Original Image into the Aligned Format

We start from the **raw original mammography image** (`notebooks/assets/example/sample_original.png`), a single-channel low-energy scan that is *not* yet in the aligned format the dataloader expects.

RISS uses the aligned image format from the training pipeline (`preprocess/png_png.py`): each PNG is a horizontal A|B concatenation where the **left half is the low-energy input** and the **right half is the recombined target** (a placeholder is used at inference time). Following the exact training-time preprocessing:

1. load the original image and convert it to grayscale;
2. rescale intensities to the 0-255 range (`img / img.max() * 255`);
3. place the low-energy image into the **red** channel (green/blue = 0), matching how `png_png.py` encodes the LE modality;
4. build a zero **placeholder** for the right/target half (the target is unknown at inference and is not used by the generator);
5. concatenate `[LE | placeholder]` horizontally to produce the aligned A|B PNG and save it under `Datasets/notebook_demo/test/`.

Reproducing this exact preprocessing keeps the input on the training distribution, so the synthetic output stays faithful. For your own data, run the same steps on each raw image before inference.

In [ ]:
import numpy as np

if not original_image.exists():
    raise FileNotFoundError(f"Cannot find the raw original image at {original_image}.")


def preprocess_to_aligned(src_path):
    """Convert a raw single-channel LE image into the aligned A|B format.

    Mirrors preprocess/png_png.py: the low-energy image goes into the red
    channel, the target/placeholder half is left as zeros, and the two halves
    are concatenated horizontally into a single 3-channel PNG.
    """
    # 1) load + grayscale
    le = Image.open(src_path).convert("L")
    le_arr = np.array(le)
    # 2) rescale intensities to 0-255
    le_arr = (le_arr / np.max(le_arr) * 255).astype(np.uint8)
    zeros_channel = np.zeros_like(le_arr)
    # 3) LE -> red channel (matches png_png.py encoding of the LE modality)
    le_rgb = np.stack((le_arr, zeros_channel, zeros_channel), axis=-1)
    # 4) zero placeholder for the target/right half (unknown at inference)
    placeholder_rgb = np.stack((zeros_channel, zeros_channel, zeros_channel), axis=-1)
    # 5) concatenate [LE | placeholder] horizontally into the aligned A|B image
    aligned = np.concatenate((le_rgb, placeholder_rgb), axis=1)
    return Image.fromarray(np.uint8(aligned)), le


demo_input_dir = demo_dataset / "test"
demo_input_dir.mkdir(parents=True, exist_ok=True)
demo_input = demo_input_dir / f"{original_image.stem}_aligned.png"

aligned_img, le_gray = preprocess_to_aligned(original_image)
aligned_img.save(demo_input)
print("Raw original image:", original_image)
print("Preprocessed aligned input saved to:", demo_input)

w, h = aligned_img.size
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes[0].imshow(le_gray, cmap="gray")
axes[0].set_title("Raw original image")
axes[0].axis("off")
axes[1].imshow(aligned_img.crop((0, 0, w // 2, h)))
axes[1].set_title("Preprocessed LE (left half, red channel)")
axes[1].axis("off")
axes[2].imshow(aligned_img.crop((w // 2, 0, w, h)))
axes[2].set_title("Target placeholder (right half)")
axes[2].axis("off")
plt.tight_layout()

## 4. Run RISS Inference

The recommended path is to call the repository `test.py` entry point, because it reuses the same model creation and preprocessing logic as the command-line workflow.

If the cell reports that `test.py` or weights are missing, download the pretrained weights and place them at:

```text
Checkpoints/le_re/latest_net_G.pth
```

In [ ]:
cmd = [
    sys.executable, str(test_py),
    "--dataroot", str(demo_dataset),
    "--name", experiment_name,
    "--gpu_ids", "0" if torch.cuda.is_available() else "-1",
    "--model", "resvit_one",
    "--which_model_netG", "resvit",
    "--pre_trained_transformer", "0",
    "--dataset_mode", "aligned",
    "--norm", "batch",
    "--phase", "test",
    "--output_nc", "1",
    "--input_nc", "3",
    "--how_many", "1",
    "--serial_batches",
    "--fineSize", "256",
    "--loadSize", "256",
    "--results_dir", str(results_dir),
    "--checkpoints_dir", str(checkpoint_dir),
    "--which_epoch", "latest",
]

if not test_py.exists():
    raise FileNotFoundError(f"Cannot find test.py at {test_py}. Run this notebook from the RISS repository.")
if not weights_path.exists():
    raise FileNotFoundError(f"Cannot find pretrained weights at {weights_path}. Please download latest_net_G.pth first.")

print("Running command:")
print(" ".join(cmd))
completed = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(f"Inference failed with exit code {completed.returncode}")

## 5. Find and Visualize the Output

Different versions of the image-to-image translation template may save generated images under slightly different subdirectories. This cell searches the demo result directory for likely synthetic recombined images.

In [ ]:
image_exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
candidate_outputs = [
    p for p in results_dir.rglob("*")
    if p.suffix.lower() in image_exts and ("fake" in p.name.lower() or "syn" in p.name.lower() or "output" in p.name.lower())
]
if not candidate_outputs:
    candidate_outputs = [p for p in results_dir.rglob("*") if p.suffix.lower() in image_exts]

print("Found output images:")
for p in candidate_outputs[:10]:
    print(" -", p)

if not candidate_outputs:
    raise FileNotFoundError(f"No output image found under {results_dir}")

output_image = candidate_outputs[0]
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes[0].imshow(Image.open(original_image).convert("L"), cmap="gray")
axes[0].set_title("Raw original image")
axes[0].axis("off")
axes[1].imshow(Image.open(demo_input).crop((0, 0, Image.open(demo_input).size[0] // 2, Image.open(demo_input).size[1])))
axes[1].set_title("Preprocessed LE input")
axes[1].axis("off")
axes[2].imshow(Image.open(output_image), cmap="gray")
axes[2].set_title("Synthetic recombined image")
axes[2].axis("off")
plt.tight_layout()
print("Visualized output:", output_image)

## 6. Batch Inference Equivalent

For full test-set inference, use the command-line workflow below and replace `--dataroot` with your preprocessed dataset directory.

In [ ]:
print(f'''
python test.py \\
  --dataroot Datasets/your_dataset \\
  --name {experiment_name} \\
  --gpu_ids 0 \\
  --model resvit_one \\
  --which_model_netG resvit \\
  --dataset_mode aligned \\
  --norm batch \\
  --phase test \\
  --output_nc 1 \\
  --input_nc 3 \\
  --how_many 10000 \\
  --serial_batches \\
  --fineSize 256 \\
  --loadSize 256 \\
  --results_dir results/full_test \\
  --checkpoints_dir Checkpoints \\
  --which_epoch latest
''')

## Notes on Quantitative Results

This notebook is a minimal inference demonstration using example images. Quantitative values obtained from this demo, if any, may differ from manuscript-level results because the paper reports aggregate performance over full test cohorts with standardized preprocessing and statistical analysis.

To reproduce manuscript-level metrics, run inference on the complete test set and then use the evaluation scripts for PSNR, SSIM, and downstream analyses.